# GameForge3D — Phase 1: Data Collection & Preparation

**FYP 2026-2027 | NUML Dept. of Computer Science**  
**Supervisor:** Ms. Tooba Sagheer

**Steps:**
1. Mount Google Drive
2. Install Libraries
3. Load Cap3D from HuggingFace (split=test)
4. Filter gaming-relevant captions
5. Load & validate router_labels.csv
6. Train / Val / Test split
7. Visualize label distribution

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/GameForge3D/checkpoints'
DATA_DIR       = '/content/drive/MyDrive/GameForge3D/data'

for d in [f'{CHECKPOINT_DIR}/router',
          f'{CHECKPOINT_DIR}/generator',
          f'{CHECKPOINT_DIR}/texture',
          DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'Data dir      : {DATA_DIR}')

## Step 2 — Install Libraries

In [ ]:
!pip install -q datasets pandas scikit-learn matplotlib seaborn
print('Libraries ready.')

## Step 3 — Inspect Cap3D Dataset Structure

Cap3D on HuggingFace only has `split='test'` (the full 785k corpus).  
We first check available splits and column names before loading.

In [ ]:
from datasets import load_dataset, get_dataset_split_names
import pandas as pd

# Check what splits exist
try:
    splits = get_dataset_split_names('tiange/Cap3D')
    print(f'Available splits: {splits}')
except Exception as e:
    print(f'Could not get splits: {e}')
    splits = ['test']

USE_SPLIT = splits[0]   # use first available split
print(f'Using split: {USE_SPLIT}')

# Stream and preview first 5 items
cap3d = load_dataset('tiange/Cap3D', split=USE_SPLIT, streaming=True, trust_remote_code=True)

samples = []
for i, item in enumerate(cap3d):
    samples.append(item)
    if i >= 4:
        break

df_preview = pd.DataFrame(samples)
print(f'Columns : {df_preview.columns.tolist()}')
print(df_preview)

## Step 4 — Filter Gaming-Relevant Captions

Scans first 50,000 samples for Weapon / Vehicle / Creature / Prop keywords.  
> **This cell takes 5–15 minutes. Do not interrupt.**

In [ ]:
from datasets import load_dataset
import pandas as pd

WEAPON_KW   = ['sword','axe','bow','gun','rifle','blade','dagger','spear',
               'lance','mace','cannon','pistol','crossbow','wand','staff',
               'knife','whip','shield','grenade','hammer','katana','saber']

VEHICLE_KW  = ['car','truck','bike','motorcycle','ship','boat','plane',
               'aircraft','submarine','tank','helicopter','spacecraft',
               'rover','chariot','vessel','train','bus','jet','mech']

CREATURE_KW = ['dragon','monster','creature','beast','wolf','zombie',
               'goblin','demon','ghost','skeleton','giant','fairy',
               'troll','vampire','golem','robot','alien','elemental',
               'dinosaur','griffin','phoenix','serpent']

PROP_KW     = ['barrel','chest','crate','lantern','bottle','table',
               'chair','door','fountain','altar','torch','key',
               'treasure','scroll','statue','pillar','ruin','cage',
               'pot','vase','throne','campfire','well']

ALL_KW = WEAPON_KW + VEHICLE_KW + CREATURE_KW + PROP_KW

# Detect caption column name
CAPTION_COL = 'caption' if 'caption' in df_preview.columns else df_preview.columns[1]
UID_COL     = 'uid'     if 'uid'     in df_preview.columns else df_preview.columns[0]
print(f'Caption column: {CAPTION_COL} | UID column: {UID_COL}')

SCAN_LIMIT  = 50_000
gaming_samples = []

cap3d_fresh = load_dataset('tiange/Cap3D', split=USE_SPLIT, streaming=True, trust_remote_code=True)

for i, item in enumerate(cap3d_fresh):
    if i % 5000 == 0:
        print(f'  Scanned {i:,} / {SCAN_LIMIT:,} | Found: {len(gaming_samples)}')
    caption = str(item.get(CAPTION_COL, '')).lower()
    if any(kw in caption for kw in ALL_KW):
        gaming_samples.append({
            'uid':     item.get(UID_COL, ''),
            'caption': item.get(CAPTION_COL, '')
        })
    if i >= SCAN_LIMIT:
        break

df_gaming = pd.DataFrame(gaming_samples)
print(f'\nTotal gaming-relevant samples: {len(df_gaming)}')
df_gaming.to_csv(f'{DATA_DIR}/cap3d_gaming_subset.csv', index=False)
print(f'Saved: {DATA_DIR}/cap3d_gaming_subset.csv')
df_gaming.head()

## Step 5 — Load & Validate router_labels.csv from GitHub

In [ ]:
import subprocess, os

REPO_PATH = '/content/GameForge3D'

if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone',
                    'https://github.com/Zubair-471/GameForge3D.git',
                    REPO_PATH], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull'], check=True)
    print('Repo updated.')

ROUTER_CSV = f'{REPO_PATH}/data/router_labels.csv'
df_router  = pd.read_csv(ROUTER_CSV)

print(f'\nRouter dataset shape : {df_router.shape}')
print('\nLabel distribution:')
print(df_router['label'].value_counts())

## Step 6 — Train / Val / Test Split (70 / 15 / 15)

In [ ]:
from sklearn.model_selection import train_test_split

df_trainval, df_test = train_test_split(
    df_router, test_size=0.15, random_state=42, stratify=df_router['label'])

df_train, df_val = train_test_split(
    df_trainval, test_size=0.176, random_state=42, stratify=df_trainval['label'])

print(f'Train : {len(df_train)}')
print(f'Val   : {len(df_val)}')
print(f'Test  : {len(df_test)}')

df_train.to_csv(f'{DATA_DIR}/router_train.csv', index=False)
df_val.to_csv(  f'{DATA_DIR}/router_val.csv',   index=False)
df_test.to_csv( f'{DATA_DIR}/router_test.csv',  index=False)
print(f'Splits saved to {DATA_DIR}')

## Step 7 — Label Distribution Chart

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
splits_dict = {'Train': df_train, 'Val': df_val, 'Test': df_test}

for ax, (name, df) in zip(axes, splits_dict.items()):
    counts = df['label'].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='viridis')
    ax.set_title(f'{name} ({len(df)} samples)')
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.5, str(v), ha='center')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/label_distribution.png', dpi=150)
plt.show()
print('Chart saved.')

## Phase 1 Complete! ✅

| Output | Location |
|--------|----------|
| Cap3D Gaming Subset | `data/cap3d_gaming_subset.csv` |
| Router Train | `data/router_train.csv` |
| Router Val | `data/router_val.csv` |
| Router Test | `data/router_test.csv` |
| Chart | `data/label_distribution.png` |

**Next → Phase 2: Asset Category Router (DistilBERT fine-tuning)**